# Setup

In [ ]:
%pip install gdown -q
!gdown --folder 16-H_fNmBLh4Vpu86xQ5DhFe87mqK5PEq -q
%pip install mteb -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 29.0 MB/s eta 0:00:00


# Lib

In [ ]:
import os
import json
import pandas as pd
import torch
import gc
import mteb
import json
import sys
import glob
from datasets import Dataset, DatasetDict
from mteb import MTEB
from mteb.abstasks import AbsTaskRetrieval, AbsTaskSTS
from mteb.abstasks.task_metadata import TaskMetadata
from sentence_transformers import SentenceTransformer

sys.path.append('content/data')
from data import retrieval

# Embedding Models

Tiêu chí lựa chọn mô hình ban đầu:
1. Xếp hạng cao trong MTEB Leaderboard/RTEB Finance
2. Có thể triển khai ở local system mà không cần thông qua api -> models open-source
3. Max token: từ 30K
4. Params: 0.6B - 4B (ưu tiên 4B) - vì hạn chế của T4
5. Có hỗ trợ eng và vie
6. Có lượt tải xuống cao từ HuggingFace

Kết quả:
- Octen-Embedding-4B
- BAAI/bge-m3
- Qwen3-Embedding-4B
- GritLM-7B
- jina-embeddings-v4 (4B)

In [ ]:
# mteb Tasks
class BankingRetrievalTask(AbsTaskRetrieval):
    metadata = TaskMetadata(
        name="BankingRetrieval",
        description="Custom Retrieval Evaluation for Banking Chatbot",
        reference=None,
        type="Retrieval",
        category="t2t",               # sentence to paragraph
        eval_splits=["test"],         # The split we are using
        eval_langs=["eng-Latn"],      # Language (use standard BCP-47 codes like 'eng-Latn' or 'vie-Latn')
        main_score="ndcg_at_10",      # Primary metric
        dataset={
            "path": "local",
            "revision": "1.0"
        }
    )
    # Implement data loading methods here to return corpus, queries, qrels
    def load_data(self, **kwargs):
        """
        MTEB calls this method to load data before evaluation.
        We map our imported dictionaries to the 'test' split.
        """
        if self.data_loaded:
            return

        # MTEB requires these exact variable names: corpus, queries, relevant_docs
        self.corpus = {"test": retrieval.corpus}
        self.queries = {"test": retrieval.queries}
        self.relevant_docs = {"test": retrieval.qrels}

        self.data_loaded = True

class BankingSTSTask(AbsTaskSTS):
    metadata = TaskMetadata(
        name="BankingSTS",
        description="Custom STS Evaluation for Banking Chatbot",
        reference=None,
        type="STS",
        category="t2t",
        eval_splits=["test"],
        eval_langs=["eng-Latn"],
        main_score="cosine_spearman",
        dataset={
            "path": "local",
            "revision": "1.0"
        }
    )
    # Implement data loading methods here to return the sentence pairs
    def load_data(self, **kwargs):
        """
        Reads the local JSON file and converts it into a Hugging Face DatasetDict
        so MTEB can seamlessly calculate correlation metrics.
        """
        if self.data_loaded:
            return

        # Load the raw JSON data
        with open("data/sts.json", "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        # Convert the list of dictionaries to a Hugging Face Dataset
        hf_dataset = Dataset.from_list(raw_data)

        # Assign it to self.dataset under the 'test' split
        self.dataset = DatasetDict({
            "test": hf_dataset
        })

        self.data_loaded = True


In [ ]:
# Define your list of models
models_to_test = [
    "BAAI/bge-m3",
    "jinaai/jina-embeddings-v4",
    "Octen/Octen-Embedding-4B",
    "Octen/Octen-Embedding-4B-INT8",
    "Qwen/Qwen3-Embedding-4B",
]

for model_name in models_to_test:
    print(f"\n{'='*40}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*40}")

    try:
        # 1. Load the model
        # (For 7B/8B models on a T4, you would need to add model_kwargs={"load_in_4bit": True})
        model = SentenceTransformer(model_name, trust_remote_code = True)

        # 2. Initialize MTEB with your tasks
        evaluation = MTEB(tasks=[BankingRetrievalTask(), BankingSTSTask()])

        # 3. Run and save to a folder specific to the model's name
        # We clean the model name so it makes a valid folder path
        evaluation.run(model, output_folder=f"results/")

        print(f"Successfully finished {model_name}")

    except Exception as e:
        print(f"Failed to evaluate {model_name}. Error: {e}")

    finally:
        # 4. CRITICAL: Destroy the model and clear GPU memory before the next loop
        if 'model' in locals():
            del model
        gc.collect()
        torch.cuda.empty_cache()


Evaluating: BAAI/bge-m3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

... (89 outputs left out)

- BankingRetrieval, t2t

STS

- BankingSTS, t2t

Successfully finished GritLM/GritLM-7B


In [ ]:
results_dir = "results"
summary_data = []

# Walk through all output folders
for model_folder in os.listdir(results_dir):
    model_path = os.path.join(results_dir, model_folder)

    if os.path.isdir(model_path):
        model_metrics = {"Model": model_folder}

        # Extract Retrieval Score (NDCG@10)
        retrieval_files = glob.glob(os.path.join(model_path, "**", "BankingRetrieval.json"), recursive=True)
        if retrieval_files: retrieval_file = retrieval_files[0]
        if os.path.exists(retrieval_file):
            with open(retrieval_file, "r") as f:
                data = json.load(f)
                # MTEB nests the scores, grab the primary one
                score = data["scores"]["test"][0]["ndcg_at_10"]
                model_metrics["Retrieval (NDCG@10)"] = round(score, 4)

        # Extract STS Score (Spearman)
        sts_files = glob.glob(os.path.join(model_path, "**", "BankingSTS.json"), recursive=True)
        if sts_files: sts_file = sts_files[0]
        if os.path.exists(sts_file):
            with open(sts_file, "r") as f:
                data = json.load(f)
                score = data["scores"]["test"][0]["cosine_spearman"]
                model_metrics["STS (Spearman)"] = round(score, 4)

        summary_data.append(model_metrics)

# Create the DataFrame
df = pd.DataFrame(summary_data)

# Check if the dataframe is completely empty
if df.empty:
    print("No results found! All models likely failed (check for Out of Memory errors).")
else:
    # Safely handle missing columns if some evaluations failed
    if "Retrieval (NDCG@10)" not in df.columns:
        df["Retrieval (NDCG@10)"] = "Missing"
    if "STS (Spearman)" not in df.columns:
        df["STS (Spearman)"] = "Missing"

    # Fill any blank spots with "Failed" (e.g., if STS passed but Retrieval crashed)
    df = df.fillna("Failed")

    # Sort safely (only sort if we have actual numbers, not strings)
    if "Retrieval (NDCG@10)" in df.columns and df["Retrieval (NDCG@10)"].dtype != object:
        df = df.sort_values(by="Retrieval (NDCG@10)", ascending=False)
    elif "STS (Spearman)" in df.columns and df["STS (Spearman)"].dtype != object:
        df = df.sort_values(by="STS (Spearman)", ascending=False)

    print("\n--- MODEL COMPARISON LEADERBOARD ---")
    print(df.to_string(index=False))

    # Optional: Save to CSV to view in Excel
    df.to_csv("model_comparison.csv", index=False)


--- MODEL COMPARISON LEADERBOARD ---
                            Model  Retrieval (NDCG@10)  STS (Spearman)
                      BAAI_bge-m3               0.5706          0.8565
         Octen_Octen-Embedding-4B               0.5560          0.8648
          Qwen_Qwen3-Embedding-4B               0.5493          0.8724
                 GritLM_GritLM-7B               0.3812          0.6322
jinaai_jina-embeddings-v2-base-en               0.3705          0.5854


-> chọn model có chỉ số cân bằng giữa Retrieval và STS: Octen-Embedding-4B